In [ ]:
import base64
import html
import json
import math
import re

import geopandas as gpd
import requests
import shapely
from shapely.geometry import LineString, Point, shape
from shapely.ops import polygonize, substring

from tunnel.config import (
    BNG,
    CATCHMENT_M,
    DERIVED,
    NORTH_PORTAL,
    ROOT,
    SOUTH_PORTAL,
    WGS84,
    catchment,
    to_bng,
    tunnel_line,
)
from tunnel.data import thames
from tunnel.ownership import rotherhithe_side

# Crossing map

> The blog's two illustrations: a map of the crossing and a timeline of the tunnels under the Thames in east London.

- **`crossing_map/crossing_map.svg`:** the proposed tunnel (on the route TfL shortlisted in 2017), the tunnels
  already under this stretch of the river, and who owns the land within `CATCHMENT_M` of the Rotherhithe portal.
- **`crossing_map/timeline.svg`:** those tunnels and the others downstream, by the year each was finished.

Both are in the Centre for British Progress house style (cream ground, deep green, Merriweather and
Inter, with the fonts embedded) and match the Design canvas they were drafted on
(https://claude.ai/artifact/9o3F2t9qs8jzHVzQoGQ9hT). Everything downloaded is cached in
`crossing_map/sources/`.

The proposed route and the Thames Archway are **indicative**: drawn from written
descriptions rather than taken from a published dataset or drawing. The figure's notes say so.


In [ ]:
MAP_DIR = ROOT / "crossing_map"
SOURCES = MAP_DIR / "sources"
SOURCES.mkdir(parents=True, exist_ok=True)

# Map view: Wapping to Canary Wharf, 1400 px wide. Equirectangular, scaled for latitude.
VIEW_LON, VIEW_LAT = (-0.0625, -0.0165), (51.4945, 51.5135)
VIEW_W = 1400
_COS = math.cos(math.radians(sum(VIEW_LAT) / 2))
_K = VIEW_W / ((VIEW_LON[1] - VIEW_LON[0]) * _COS)
VIEW_H = round((VIEW_LAT[1] - VIEW_LAT[0]) * _K)
M_PER_PX = 111_320 / _K
VIEW_BOX = shapely.box(-20, -20, VIEW_W + 20, VIEW_H + 20)
VIEW_AREA = gpd.GeoSeries([shapely.box(VIEW_LON[0], VIEW_LAT[0], VIEW_LON[1], VIEW_LAT[1])], crs=WGS84).to_crs(BNG)[0]
OSM_BBOX = "51.488,-0.070,51.518,-0.006"  # a little beyond the view

# Centre for British Progress palette
CREAM, INK, GREEN, SLATE, TEXT, PALE = "#FFFBEA", "#0A4938", "#3A8D5D", "#4E6464", "#293130", "#DCEBDD"
LAND, WATER, PARK, ROAD = "#F3EED8", "#CFE0E6", "#E1E6CC", "#FFFDF6"
SANS, SERIF = "Inter, Arial, sans-serif", "Merriweather, Georgia, serif"
FONTS_URL = "https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Merriweather:ital,wght@0,300;1,300"
BROWSER_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
)


def cached(path, url: str) -> str:
    if not path.exists():
        r = requests.get(url, headers={"User-Agent": "tunnel-stuff crossing map"}, timeout=120)
        r.raise_for_status()
        path.write_text(r.text)
    return path.read_text()


def overpass_url(query: str) -> str:
    return "https://overpass.kumi.systems/api/interpreter?data=" + requests.utils.quote(query)


def ll(lon, lat):
    return gpd.GeoSeries([Point(lon, lat)], crs=WGS84).to_crs(BNG)[0].coords[0]


def px(geom, crs=BNG):
    """Geometry to map pixels."""
    if crs != WGS84:
        geom = gpd.GeoSeries([geom], crs=crs).to_crs(WGS84)[0]
    return shapely.ops.transform(lambda x, y, z=None: ((x - VIEW_LON[0]) * _COS * _K, (VIEW_LAT[1] - y) * _K), geom)


def svg_d(geom) -> str:
    rings = []
    for part in shapely.get_parts(geom):
        if part.geom_type == "Polygon":
            rings += [part.exterior, *part.interiors]
        elif part.geom_type in ("LineString", "LinearRing"):
            rings.append(part)
    return "".join(
        "M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in r.coords) + ("Z" if r.geom_type == "LinearRing" else "")
        for r in rings
    )


def text(x, y, s, size, weight=400, fill=TEXT, family=SANS, anchor="start", extra="") -> str:
    """One line of text; y is the top of the line box (as on the canvas), converted to a baseline."""
    return (
        f'<text x="{x:.1f}" y="{y + size * 0.95:.1f}" font-family="{family}" font-size="{size}" '
        f'font-weight="{weight}" fill="{fill}" text-anchor="{anchor}" {extra}>{html.escape(s)}</text>'
    )


def embedded_fonts() -> str:
    """@font-face rules for the Latin subsets, with the woff2 files inlined, so the SVGs render the
    same in an <img> tag."""
    css_path = SOURCES / "fonts.css"
    if not css_path.exists():  # Google serves woff2 only to browsers it recognises
        css_path.write_text(requests.get(FONTS_URL, headers={"User-Agent": BROWSER_UA}, timeout=60).text)
    faces = {}
    for block in re.findall(r"/\* latin \*/\s*@font-face\s*{([^}]*)}", css_path.read_text()):
        family, style, weight, url = (
            re.search(pat, block).group(1)
            for pat in (r"font-family: '([^']+)'", r"font-style: (\w+)", r"font-weight: (\d+)", r"url\(([^)]+)\)")
        )
        faces.setdefault((family, style, url), []).append(int(weight))
    rules = []
    for (family, style, url), weights in faces.items():
        font = SOURCES / f"font_{url.rsplit('/', 1)[-1]}"
        if not font.exists():
            font.write_bytes(requests.get(url, timeout=60).content)
        b64 = base64.b64encode(font.read_bytes()).decode()
        rules.append(
            f"@font-face{{font-family:'{family}';font-style:{style};font-weight:{min(weights)} {max(weights)};"
            f"src:url(data:font/woff2;base64,{b64}) format('woff2')}}"
        )
    return f"<style>{''.join(rules)}</style>"


def card(left, top, width, title, lines, dark=False, large=False, align="start") -> list[str]:
    """A label box: bold title over detail lines (broken by hand, as SVG text doesn't wrap).
    Dark boxes are for the proposed tunnel."""
    pad_x, pad_y, title_size, line_size = (14, 12, 16, 12.5) if large else (10, 8, 13, 12)
    title_h, line_h = title_size * 1.3, line_size * 1.42
    height = pad_y * 2 + title_h + 3 + line_h * len(lines)
    x = left + pad_x if align == "start" else left + width - pad_x
    bg = f'fill="{INK}"' if dark else f'fill="{CREAM}" fill-opacity="0.94"'
    out = [f'<rect x="{left}" y="{top}" width="{width}" height="{height:.0f}" {bg}/>']
    out.append(text(x, top + pad_y + 1, title, title_size, 600, CREAM if dark else TEXT, anchor=align))
    for i, line in enumerate(lines):
        y = top + pad_y + title_h + 3 + i * line_h + 2
        out.append(text(x, y, line, line_size, 400, PALE if dark else SLATE, anchor=align))
    return out

## Land ownership

Title plans in the readjustment area, from `data/derived/catchment_plots.gpkg` (written by the ownership
notebook), clipped to the Rotherhithe side of the river centre line. Plots mostly covered by a larger
one (sub-titles, leases) are dropped, and so is the Port of London Authority's river bed and foreshore,
so the river reads as water.


In [ ]:
own = gpd.read_file(DERIVED / "catchment_plots.gpkg")
own = gpd.clip(own[own.top_level], rotherhithe_side(catchment()))
own["geometry"] = own.geometry.apply(
    lambda g: shapely.union_all([p for p in shapely.get_parts(g) if p.geom_type == "Polygon"])
)
own = own[~own.geometry.is_empty & ~own.address.fillna("").str.lower().str.contains("bed and foreshore")]
print(own.group.value_counts().to_string())

group
private      1292
other         210
housing        74
southwark      55


## Tunnels

Each tunnel on the map is one line in `LINES` and its portals or shafts are points in `POINTS`, all in BNG.

- **Rail:** the Jubilee, Elizabeth and East London lines wherever they run underground: the line comes
  from Oliver O'Brien's `tfl_lines.json`, kept where OpenStreetMap tags the track as tunnel (gaps of up
  to 400 m, mostly underground stations, are filled). Brunel's Thames Tunnel is drawn on its own.
- **Rotherhithe road tunnel:** straight from OpenStreetMap.
- **Thames Archway** (1805–08): the abandoned driftway, inferred from descriptions.
- **Thames Tideway Tunnel:** Tideway hasn't published its alignment, but the City of London's INSPIRE map
  service carries the planning safeguarding corridor (about 28 m wide). The line is that corridor's
  centre line, so within about 14 m of the safeguarded route, with shaft sites joined straight across.
- **Proposed tunnel:** the immersed tube TfL shortlisted in 2017. TfL published no route, only that it "could be
  relatively flexible in location"; IanVisits' write-up of the consultation lands it in two docks, Nelson Dock and
  the West India Docks impound lock inlet. The line, with its ramps folded inside each dock, comes from the config.


In [ ]:
LINES, POINTS = {}, {}

# Rail
TFL_LINES_URL = "https://raw.githubusercontent.com/oobrien/vis/master/tubecreature/data/tfl_lines.json"
RAIL_TUNNELS_URL = overpass_url(
    f'[out:json][timeout:120];way["railway"~"^(subway|rail|light_rail)$"]["tunnel"="yes"]({OSM_BBOX});out tags geom;'
)


def rail_tunnel_ways(pattern: str):
    """OpenStreetMap railway tunnel ways whose name matches `pattern`, in BNG."""
    ways = json.loads(cached(SOURCES / "rail_tunnels_osm.json", RAIL_TUNNELS_URL))["elements"]
    return shapely.union_all(
        [
            LineString([ll(pt["lon"], pt["lat"]) for pt in e["geometry"]])
            for e in ways
            if re.search(pattern, e["tags"].get("name", ""))
        ]
    )


def tunnel_sections(line_name: str, pattern: str, max_gap: float = 400, min_len: float = 200):
    """The line's route where OpenStreetMap tags the track as tunnel, with gaps of up to
    `max_gap` metres (underground stations, crossovers) filled in. Gaps under the Thames are
    always filled: TfL's route is schematic there and can stray 100 m+ from the bores."""
    route = [
        shape(f["geometry"])
        for f in json.loads(cached(SOURCES / "tfl_lines.json", TFL_LINES_URL))["features"]
        if any(line["name"] == line_name for line in f["properties"]["lines"])
    ]
    route = gpd.GeoSeries(route, crs=WGS84).to_crs(BNG).union_all()
    route = shapely.line_merge(route.intersection(VIEW_AREA.buffer(2000)))
    near_tunnel = rail_tunnel_ways(pattern).buffer(40)
    river = thames().union_all()
    sections = []
    for r in shapely.get_parts(route):
        spans = sorted(
            sorted(r.project(Point(c)) for c in (part.coords[0], part.coords[-1]))
            for part in shapely.get_parts(r.intersection(near_tunnel))
            if part.geom_type == "LineString" and not part.is_empty
        )
        merged = []
        for a, b in spans:
            if merged and (a - merged[-1][1] <= max_gap or substring(r, merged[-1][1], a).intersects(river)):
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        sections += [substring(r, a, b) for a, b in merged if b - a >= min_len]
    return shapely.union_all(sections)


LINES["Jubilee line"] = tunnel_sections("Jubilee", r"Jubilee")
LINES["Elizabeth line"] = tunnel_sections("Elizabeth line", r"Elizabeth")
east_london = tunnel_sections("London Overground", r"East London|Thames Tunnel")
brunel = rail_tunnel_ways(r"^Thames Tunnel$").buffer(40)
LINES["Thames Tunnel"] = east_london.intersection(brunel)
LINES["East London line"] = east_london.difference(brunel)

# Rotherhithe road tunnel
ROTHERHITHE_URL = overpass_url(
    f'[out:json][timeout:30];way["tunnel"="yes"]["name"="Rotherhithe Tunnel"]({OSM_BBOX});out geom tags;'
)
(way,) = json.loads(cached(SOURCES / "rotherhithe_tunnel_osm.json", ROTHERHITHE_URL))["elements"]
LINES["Rotherhithe Tunnel"] = LineString([ll(pt["lon"], pt["lat"]) for pt in way["geometry"]])

# Thames Archway driftway (1805-08): shaft near the end of Lavender Street, aiming for the Regent's
# Canal Dock entrance at Limehouse. It got within about 117 ft of the north bank before flooding ended it.
shaft, target = ll(-0.0378, 51.5068), ll(-0.0365, 51.5093)
driftway = LineString([shaft, target])
LINES["Thames Archway"] = LineString([shaft, driftway.interpolate(driftway.length - 36)])
POINTS["Thames Archway shaft"] = Point(shaft)

# Thames Tideway Tunnel: centre line of the safeguarding corridor, near the view
TIDEWAY_URL = (
    "https://www.mapping.cityoflondon.gov.uk/arcgis/rest/services/INSPIRE/MapServer/129/query"
    "?where=1%3D1&outFields=*&outSR=27700&f=geojson"
)


def centreline(corridor, spacing: float = 5.0):
    """Medial axis of a long thin polygon: the Voronoi edges of its outline that lie wholly inside it."""
    outline = shapely.MultiPoint(shapely.get_coordinates(shapely.segmentize(corridor.boundary, spacing)))
    inside = corridor.buffer(-0.5)
    edges = [e for e in shapely.get_parts(shapely.voronoi_polygons(outline, only_edges=True)) if inside.contains(e)]
    return shapely.line_merge(shapely.union_all(edges)).simplify(2)


def join_ends(parts: list, max_gap: float = 400) -> list:
    """Straight links across shaft sites: from each loose end to the nearest loose end of another part."""
    ends = [(i, Point(p.coords[k])) for i, p in enumerate(parts) for k in (0, -1)]
    loose = [(i, pt) for i, pt in ends if not any(j != i and pt.distance(q) < 0.5 for j, q in ends)]
    links = []
    for i, pt in loose:
        gap, q = min(((pt.distance(q), q) for j, q in loose if j != i), key=lambda t: t[0])
        if gap <= max_gap:
            links.append(LineString([pt, q]))
    return links


safeguarding = gpd.read_file(cached(SOURCES / "tideway_safeguarding.geojson", TIDEWAY_URL))
corridors = (
    safeguarding[safeguarding.TYPE == "Safeguarding for tunnels"].union_all().intersection(VIEW_AREA.buffer(1000))
)
# slivers under 1000 m2 are drawing artefacts
parts = [
    p for c in shapely.get_parts(corridors) if c.area > 1000 for p in shapely.get_parts(centreline(c)) if p.length > 20
]
LINES["Tideway Tunnel"] = shapely.line_merge(shapely.union_all(parts + join_ends(parts)))

# Proposed tunnel
LINES["Proposed tunnel"] = tunnel_line()
POINTS["Proposed south portal"] = Point(to_bng(SOUTH_PORTAL))
POINTS["Proposed north portal"] = Point(to_bng(NORTH_PORTAL))

for name, line in LINES.items():
    print(f"  {name}: {line.intersection(VIEW_AREA).length / 1000:.2f} km in view")

  Jubilee line: 3.39 km in view
  Elizabeth line: 1.81 km in view
  Thames Tunnel: 0.57 km in view
  East London line: 1.67 km in view
  Rotherhithe Tunnel: 1.48 km in view
  Thames Archway: 0.26 km in view
  Tideway Tunnel: 3.63 km in view
  Proposed tunnel: 1.00 km in view


## Basemap

River, docks, parks and main roads from OpenStreetMap, then the ownership plots, the 750 m ring and the
tunnels. The anchors printed at the end are where each tunnel crosses the river, in map pixels: the
labels in the next section point at them.


In [ ]:
BASEMAP_OSM_URL = overpass_url(
    "[out:json][timeout:120];("
    f'nwr["natural"="water"]({OSM_BBOX});'
    f'nwr["leisure"="park"]({OSM_BBOX});'
    f'way["highway"~"^(primary|secondary|trunk|tertiary)$"]({OSM_BBOX});'
    ");out geom;"
)

OWNER_STYLES = {  # group -> (legend label, fill, outline), drawn in this order
    "private": ("Private individuals", "#EAE2C8", "#D3C7A4"),
    "other": ("Other companies and bodies", "#DDC791", "#B9A066"),
    "housing": ("Housing associations", "#A7C7EE", "#6F9FD6"),
    "southwark": ("Southwark Council", "#9FCBA9", "#5F9F78"),
}
# LINES key -> (colour, width px, dash, cream casing), bottom to top
LINE_STYLES = {
    "Tideway Tunnel": ("#7FA3A0", 3.5, "10 4", False),
    "Elizabeth line": ("#6950A1", 3, None, True),
    "East London line": ("#EE7C0E", 3, None, True),
    "Thames Tunnel": ("#EE7C0E", 4.5, None, True),
    "Jubilee line": ("#7B868C", 3, None, True),
    "Rotherhithe Tunnel": (TEXT, 4.5, None, True),
    "Thames Archway": ("#8C6D31", 2.5, "0.1 6", False),
    "Proposed tunnel": (INK, 7, None, True),
}
# POINTS key -> (radius, outline colour)
POINT_STYLES = {
    "Thames Archway shaft": (4, "#8C6D31"),
    "Proposed south portal": (7, INK),
    "Proposed north portal": (7, INK),
}


def osm_area(e):
    """Polygon for a closed OSM way or a multipolygon relation, else None."""
    if e["type"] == "node":
        return None
    if e["type"] == "way":
        pts = [(p["lon"], p["lat"]) for p in e["geometry"]]
        return shapely.Polygon(pts).buffer(0) if len(pts) > 3 and pts[0] == pts[-1] else None
    rings = {
        role: shapely.union_all(
            list(
                polygonize(
                    shapely.union_all(
                        [
                            LineString([(p["lon"], p["lat"]) for p in m["geometry"]])
                            for m in e["members"]
                            if m.get("role") == role and m.get("geometry")
                        ]
                    )
                )
            )
        )
        for role in ("outer", "inner")
    }
    return rings["outer"].difference(rings["inner"])


water, parks, roads = [], [], {}
for e in json.loads(cached(SOURCES / "basemap_osm.json", BASEMAP_OSM_URL))["elements"]:
    t = e.get("tags", {})
    if t.get("natural") == "water" and t.get("water") not in ("fountain", "pond"):
        water.append(osm_area(e))
    elif t.get("leisure") == "park" or t.get("natural") == "wood":
        parks.append(osm_area(e))
    elif "highway" in t and e["type"] == "way":
        roads.setdefault(t["highway"], []).append(LineString([(p["lon"], p["lat"]) for p in e["geometry"]]))

water = px(shapely.union_all([g for g in water if g is not None]), WGS84).intersection(VIEW_BOX).simplify(0.4)
parks = px(shapely.union_all([g for g in parks if g is not None and g.area > 2e-8]), WGS84)
parks = parks.intersection(VIEW_BOX).difference(water).simplify(0.4)
lines_px = {name: px(line).intersection(VIEW_BOX) for name, line in LINES.items()}
points_px = {name: px(pt) for name, pt in POINTS.items()}

basemap = [
    f'<rect width="{VIEW_W}" height="{VIEW_H}" fill="{LAND}"/>',
    f'<path d="{svg_d(parks)}" fill="{PARK}"/>',
    f'<path d="{svg_d(water)}" fill="{WATER}" fill-rule="evenodd"/>',
]
for group, (_, fill, outline) in OWNER_STYLES.items():
    plots = px(own[own.group == group].geometry.buffer(0).union_all()).intersection(VIEW_BOX).simplify(0.3)
    basemap.append(
        f'<path d="{svg_d(plots)}" fill="{fill}" stroke="{outline}" stroke-width="0.6" '
        'stroke-linejoin="round" fill-rule="evenodd"/>'
    )
for kind, width in (("tertiary", 1.6), ("secondary", 2.4), ("primary", 3.0), ("trunk", 3.4)):
    d = svg_d(px(shapely.union_all(roads.get(kind, [])), WGS84).intersection(VIEW_BOX).simplify(0.4))
    basemap.append(
        f'<path d="{d}" fill="none" stroke="{ROAD}" stroke-width="{width}" '
        'stroke-linecap="round" stroke-linejoin="round" opacity="0.95"/>'
    )
ring = points_px["Proposed south portal"].buffer(CATCHMENT_M / M_PER_PX, quad_segs=32).exterior
basemap.append(
    f'<path d="{svg_d(ring)}" fill="none" stroke="{INK}" stroke-width="1.5" '
    'stroke-dasharray="1 6" stroke-linecap="round" opacity="0.8"/>'
)
for name, (colour, width, dash, casing) in LINE_STYLES.items():
    d = svg_d(lines_px[name])
    if casing:
        basemap.append(
            f'<path d="{d}" fill="none" stroke="{CREAM}" stroke-width="{width + 4}" '
            'stroke-linecap="round" stroke-linejoin="round" opacity="0.9"/>'
        )
    dash_attr = f' stroke-dasharray="{dash}"' if dash else ""
    basemap.append(
        f'<path d="{d}" fill="none" stroke="{colour}" stroke-width="{width}" '
        f'stroke-linecap="round" stroke-linejoin="round"{dash_attr}/>'
    )
for name, (r, outline) in POINT_STYLES.items():
    pt = points_px[name]
    basemap.append(
        f'<circle cx="{pt.x:.0f}" cy="{pt.y:.0f}" r="{r}" fill="{CREAM}" stroke="{outline}" stroke-width="2.5"/>'
    )

print(f"Map: {VIEW_W} x {VIEW_H} px, {M_PER_PX:.3f} m/px. Where each tunnel crosses the river:")
river_px = max(shapely.get_parts(water), key=lambda p: p.area)
for name, line in lines_px.items():
    crossing = line.intersection(river_px)
    if not crossing.is_empty:
        print(f"  {name}: {crossing.centroid.x:.0f}, {crossing.centroid.y:.0f}")

Map: 1400 x 929 px, 2.277 m/px. Where each tunnel crosses the river:
  Jubilee line: 986, 479
  Elizabeth line: 857, 73
  Thames Tunnel: 244, 509
  Rotherhithe Tunnel: 423, 351
  Thames Archway: 773, 263
  Tideway Tunnel: 433, 334
  Proposed tunnel: 1027, 489


## Map figure

The basemap with the tunnel labels, place names, scale bar, legend and sources. Label text,
line breaks and positions are set by hand in `CARDS` and `PLACES`, as on the canvas: move them if a
change to the data makes them collide.


In [ ]:
FIGURE = MAP_DIR / "crossing_map.svg"
FIG_W, FIG_H, MAP_TOP = VIEW_W, VIEW_H + 269, 0

# (left, top, width, title, detail lines, leader from anchor (x, y) to (x, y)), in map px
CARDS = [
    (24, 792, 210, "Jubilee line", ["1999 · tube trains, Canada Water", "to Canary Wharf"], (120, 768, 120, 792)),
    (
        24,
        640,
        200,
        "Thames Tunnel",
        ["1843 · a foot tunnel until 1865,", "now London Overground"],
        (244, 511, 160, 640),
    ),
    (
        196,
        228,
        200,
        "Rotherhithe Tunnel",
        ["1908 · road, shared with", "cyclists and pedestrians"],
        (423, 349, 396, 282),
    ),
    (560, 226, 180, "Thames Archway", ["1805–08 · pilot tunnel,", "abandoned after floods"], (775, 262, 740, 252)),
    (1150, 240, 196, "Elizabeth line", ["2022 · Canary Wharf station"], (1180, 310, 1180, 284)),
    (16, 418, 160, "Tideway Tunnel", ["2025 · storm sewer"], (110, 598, 100, 474)),
]
HERO = (
    1130,
    556,
    262,
    "Proposed tunnel",
    [
        "TfL’s 2017 immersed tube route,",
        "Nelson Dock to the impound lock:",
        "384 m under the river, ramps folded",
        "in each dock, about 940 m end to end.",
        "A 3.5 m cycle lane and 2.5 m footway.",
    ],
    (1060, 499, 1130, 574),
)
# (text, left, top) in map px
PLACES = [
    ("WAPPING", 70, 334),
    ("LIMEHOUSE", 420, 66),
    ("ROTHERHITHE", 446, 520),
    ("CANARY WHARF", 1210, 192),
    ("ISLE OF DOGS", 1180, 790),
    ("CANADA WATER", 300, 866),
]
CREAM_HALO = f'paint-order="stroke" stroke="{CREAM}" stroke-width="4" stroke-linejoin="round"'
LAND_HALO = f'paint-order="stroke" stroke="{LAND}" stroke-width="4" stroke-linejoin="round"'


def swatch_line(x, y, colour, width, dash=None, w=40) -> str:
    dash_attr = f' stroke-dasharray="{dash}" stroke-linecap="round"' if dash else ""
    return (
        f'<line x1="{x + 2}" y1="{y}" x2="{x + w - 2}" y2="{y}" stroke="{colour}" stroke-width="{width}"{dash_attr}/>'
    )


fig = [
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{FIG_W}" height="{FIG_H}" viewBox="0 0 {FIG_W} {FIG_H}">',
    f"<title>Rotherhithe to Canary Wharf crossing map</title>{embedded_fonts()}",
    f'<rect width="{FIG_W}" height="{FIG_H}" fill="{CREAM}"/>',
    # Map
    f'<clipPath id="map"><rect width="{VIEW_W}" height="{VIEW_H}"/></clipPath>',
    f'<g transform="translate(0 {MAP_TOP})" clip-path="url(#map)">',
    *basemap,
]
for *_, (x1, y1, x2, y2) in [*CARDS, HERO]:
    fig.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{TEXT}" stroke-width="1"/>')
    fig.append(f'<circle cx="{x1}" cy="{y1}" r="2.5" fill="{TEXT}"/>')
for left, top, width, title, lines, _ in CARDS:
    fig += card(left, top, width, title, lines)
fig += card(*HERO[:5], dark=True, large=True)
fig += [
    text(860, 440, "Nelson Dock", 12, 600, INK, extra=CREAM_HALO),
    text(1122, 518, "Impound lock", 12, 600, INK, extra=CREAM_HALO),
    text(914, 226, f"{CATCHMENT_M} m", 11, 500, INK),
    text(
        0,
        -8,
        "River Thames",
        15,
        300,
        SLATE,
        SERIF,
        "middle",
        'font-style="italic" transform="translate(985 752) rotate(90)"',
    ),
    text(
        0,
        -8,
        "River Thames",
        15,
        300,
        SLATE,
        SERIF,
        "start",
        'font-style="italic" transform="translate(300 480) rotate(-38)"',
    ),
    *(text(x, y, s, 11, 600, SLATE, extra=f'letter-spacing="1.76" {LAND_HALO}') for s, x, y in PLACES),
    # Scale bar (500 m) and north arrow
    f'<rect x="1124" y="862" width="256" height="52" fill="{CREAM}" fill-opacity="0.94"/>',
    f'<rect x="1136" y="874" width="{250 / M_PER_PX:.1f}" height="5" fill="{INK}"/>',
    f'<rect x="{1136 + 250 / M_PER_PX:.1f}" y="874" width="{250 / M_PER_PX:.1f}" height="5" fill="none" stroke="{INK}" stroke-width="1"/>',
    text(1136, 886, "0", 11, 400, SLATE),
    text(1136 + 250 / M_PER_PX, 886, "250", 11, 400, SLATE, anchor="middle"),
    text(1136 + 500 / M_PER_PX, 886, "500 m", 11, 400, SLATE, anchor="end"),
    f'<path d="M1379 870 L1384 884 L1379 881 L1374 884 Z" fill="{INK}"/>',
    text(1379, 888, "N", 10, 600, INK, anchor="middle"),
    "</g>",
    f'<line x1="0" y1="{MAP_TOP + VIEW_H}" x2="{FIG_W}" y2="{MAP_TOP + VIEW_H}" stroke="{INK}"/>',
]

# Legend
top = MAP_TOP + VIEW_H + 38
heading = lambda x, s: text(x, top, s, 12, 600, GREEN, extra='letter-spacing="1.68"')
row_y = lambda i: top + 30 + i * 26  # top of each legend row
fig.append(heading(64, "CROSSINGS"))
crossings = [
    ("Proposed walking and cycling tunnel", [swatch_line(64, 0, INK, 6)]),
    (
        "Rail: Overground, Jubilee, Elizabeth line",
        [swatch_line(64, -4, "#EE7C0E", 3), swatch_line(64, 0, "#7B868C", 3), swatch_line(64, 4, "#6950A1", 3)],
    ),
    ("Road tunnel", [swatch_line(64, 0, TEXT, 4.5)]),
    ("Sewer tunnel", [swatch_line(64, 0, "#7FA3A0", 3.5, "10 4")]),
    ("Abandoned tunnel", [swatch_line(64, 0, "#8C6D31", 2.5, "0.1 6")]),
]
for i, (label, marks) in enumerate(crossings):
    mid = row_y(i) + 8
    fig.append(f'<g transform="translate(0 {mid})">{"".join(marks)}</g>')
    if i == 0:
        fig += [
            f'<circle cx="{x}" cy="{mid}" r="4.5" fill="{CREAM}" stroke="{INK}" stroke-width="2.5"/>' for x in (70, 98)
        ]
    fig.append(text(116, row_y(i), label, 13.5))

fig.append(heading(504, f"LAND WITHIN {CATCHMENT_M} M OF THE ROTHERHITHE PORTAL"))
for i, group in enumerate(reversed(OWNER_STYLES)):
    label, fill, outline = OWNER_STYLES[group]
    fig += [
        f'<rect x="504.5" y="{row_y(i) + 1.5}" width="22" height="14" fill="{fill}" stroke="{outline}"/>',
        text(540, row_y(i), label, 13.5),
        text(896, row_y(i), f"{(own.group == group).sum():,} titles", 13.5, 400, SLATE, anchor="end"),
    ]
fig += [
    f'<g transform="translate(0 {row_y(4) + 8})">{swatch_line(502, 0, INK, 1.5, "1 5", 24)}</g>',
    text(540, row_y(4), f"{CATCHMENT_M} m from the south portal", 13.5),
    heading(944, "NOTES"),
]
NOTES = [
    "The proposed route and the Thames Archway are",
    "indicative, drawn from written descriptions rather than",
    "published drawings. The river bed, owned by the Port of",
    "London Authority, is not shaded.",
]
SOURCES_TEXT = [
    "Sources: HM Land Registry NPS, CCOD and OCOD, © Crown",
    "copyright, OGL v3.0. Map data © OpenStreetMap contributors,",
    "ODbL. TfL 2017 consultation; IanVisits; Wikipedia.",
]
fig += [text(944, top + 30 + i * 20, s, 13) for i, s in enumerate(NOTES)]
fig += [text(944, top + 30 + 20 * len(NOTES) + 10 + i * 18.6, s, 12, 400, SLATE) for i, s in enumerate(SOURCES_TEXT)]
fig.append("</svg>")
FIGURE.write_text("\n".join(fig))
print(f"{FIGURE.name}: {FIG_W} x {FIG_H} px, {FIGURE.stat().st_size / 1e3:.0f} kB")

crossing_map.svg: 1400 x 1198 px, 352 kB


## Timeline

The tunnels under the Thames in east London, by the year each was finished, ending with the proposed
tunnel. Labels above the axis sit to the right of their
stem and labels below it to the left, in tiers set so that no stem runs through another label.


In [ ]:
TIMELINE_SVG = MAP_DIR / "timeline.svg"
TL_W, TL_H, AXIS_Y = 1400, 588, 298
YEARS, AXIS_X = (1800, 2025), (64, 1260)
PROPOSED_X = 1336  # past the end of the axis: no date yet


def year_x(year: float) -> float:
    return AXIS_X[0] + (year - YEARS[0]) * (AXIS_X[1] - AXIS_X[0]) / (YEARS[1] - YEARS[0])


# (year finished, title, detail, side, tier from the axis, kind)
TIMELINE = [
    (1808, "Thames Archway", "1805–08 · abandoned", "above", 1, "abandoned"),
    (1843, "Thames Tunnel", "1825–43 · now London Overground", "above", 1, "built"),
    (1897, "Blackwall Tunnel", "1892–97 · first bore, road", "above", 4, "built"),
    (1902, "Greenwich foot tunnel", "1899–1902 · pedestrians", "above", 3, "built"),
    (1908, "Rotherhithe Tunnel", "1904–08 · road, cycles, pedestrians", "above", 2, "built"),
    (1912, "Woolwich foot tunnel", "1910–12 · pedestrians", "above", 1, "built"),
    (1967, "Blackwall Tunnel", "1960–67 · second bore, road", "above", 1, "built"),
    (1999, "Jubilee line and DLR", "1993–99 · four crossings", "below", 1, "built"),
    (2017, "TfL immersed tube option", "2017 · shortlisted, not built", "below", 2, "option"),
    (2022, "Elizabeth line", "2009–22 · trains", "below", 3, "built"),
    (2025, "Silvertown Tunnel", "2020–25 · road", "below", 4, "built"),
]
TIER_STEP, LABEL_H = 56, 46
TICK_LABEL_SHIFT = {2000: -27}  # clear of the Jubilee line's stem

tl = [
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{TL_W}" height="{TL_H}" viewBox="0 0 {TL_W} {TL_H}">',
    f"<title>Tunnels under the Thames in east London, by year</title>{embedded_fonts()}",
    f'<rect width="{TL_W}" height="{TL_H}" fill="{CREAM}"/>',
    f'<line x1="{AXIS_X[0]}" y1="{AXIS_Y}" x2="{AXIS_X[1]}" y2="{AXIS_Y}" stroke="{INK}" stroke-width="1.5"/>',
    f'<line x1="{AXIS_X[1]}" y1="{AXIS_Y}" x2="{PROPOSED_X}" y2="{AXIS_Y}" stroke="{INK}" stroke-width="1.5" stroke-dasharray="3 5"/>',
]
for year in range(YEARS[0], YEARS[1], 50):
    x = year_x(year)
    tl.append(f'<line x1="{x:.1f}" y1="{AXIS_Y - 6}" x2="{x:.1f}" y2="{AXIS_Y + 6}" stroke="{INK}"/>')
    tl.append(text(x + TICK_LABEL_SHIFT.get(year, 0), AXIS_Y + 12, str(year), 12, 400, SLATE, anchor="middle"))

labels, dots = [], []
for year, title, detail, side, tier, kind in TIMELINE:
    x = year_x(year)
    if side == "above":
        top = AXIS_Y - 90 - (tier - 1) * TIER_STEP
        stem_end, tx, anchor = top + LABEL_H, x + 8, "start"
    else:
        top = AXIS_Y + 48 + (tier - 1) * TIER_STEP
        stem_end, tx, anchor = top, x - 8, "end"
    tl.append(f'<line x1="{x:.1f}" y1="{AXIS_Y}" x2="{x:.1f}" y2="{stem_end}" stroke="#9AA9A4"/>')
    labels += [
        text(tx, top, title, 13.5, 600, anchor=anchor),
        text(tx, top + 19.5, detail, 12, 400, SLATE, anchor=anchor),
    ]
    if kind == "built":
        dots.append(f'<circle cx="{x:.1f}" cy="{AXIS_Y}" r="5.5" fill="{INK}"/>')
    else:
        ring_colour = "#8C6D31" if kind == "abandoned" else SLATE
        dots.append(
            f'<circle cx="{x:.1f}" cy="{AXIS_Y}" r="5" fill="{CREAM}" stroke="{ring_colour}" stroke-width="2"/>'
        )

# The proposed tunnel: a ring past the end of the axis, with a filled label above
tl += [
    f'<line x1="{PROPOSED_X}" y1="{AXIS_Y}" x2="{PROPOSED_X}" y2="{AXIS_Y - 30}" stroke="{INK}" stroke-width="1.5"/>',
    *dots,
    f'<circle cx="{PROPOSED_X}" cy="{AXIS_Y}" r="8" fill="{CREAM}" stroke="{INK}" stroke-width="3"/>',
    *labels,
    *card(
        PROPOSED_X - 196, AXIS_Y - 86, 196, "Proposed tunnel", ["Rotherhithe to Canary Wharf"], dark=True, align="end"
    ),
    "</svg>",
]
TIMELINE_SVG.write_text("\n".join(tl))
print(f"{TIMELINE_SVG.name}: {TL_W} x {TL_H} px, {TIMELINE_SVG.stat().st_size / 1e3:.0f} kB")

timeline.svg: 1400 x 588 px, 202 kB
